# Abstract
We analyze individual speaker performance and team success in competitive high
school debates using data from Czech debate tournaments. Using both pooled OLS
and fixed effects panel models, we find that teammate quality is the strongest
predictor of individual speaker scores, while debater experience shows a
significant positive effect. Surprisingly, we find no evidence of gender gaps,
motion balance effects, or warm-up effects in individual performance. At the
team level, neither gender composition nor average experience significantly
predicts debate outcomes.

# Introduction
We attempt to answer whether any of the following have an impact on the speaker
points gained by debaters at debate tournaments, and if so, by how much.


1. Age and Gender Effects: Do older debaters have a competitive advantage over
younger participants? Is there a measurable gender performance gap in speaker
scores?

We analyze this by estimating the gender of each debater from their first and
last name and using that as a regressor in our regressions later on.

Additionally, we also analyzed the gender effect on a per-team basis (i.e. do
teams with a higher proportion of girls score more or less ballots?).

For the age effect, we use the years since the debater's first debate as a regressor in our equation. 
This is not strictly measuring the effect of age (e.g. "18 years old"), but rather the effect
of how much debate experience one has (what we might term "debate age"). 
Given the ultimate goal of our analysis was to hopefully gather learning insights 
for debaters to improve, we believe this ended being more relevant than strictly looking at age.


2. Topic Balance: Which debate categories (economics, culture, politics) tend to
produce balanced debates versus one-sided outcomes? Does topic type correlate
with score distributions?

We analyze this by including an interaction term in our regression - i.e. we measure the effect of which side is favored by a given motion and the side that the debater actually represented in a given debate.

3. School Environment Impact: Does attending a school with high-performing
debaters improve individual performance through peer learning, or does it create
a demotivating competitive environment?

We proxy for the school effect by the scores that a speakers teammates get (which should be close enough, since most debate teams are formed by people from the same school).

4. Warm-up Effect: In multi-round tournaments, do debaters show measurable
improvement as rounds progress, suggesting a "warm-up" period?

We add the tournament round as a regressor in our speaker score equation below. 

In [1]:
import sys
import pandas as pd
from pathlib import Path
from linearmodels.panel import PanelOLS
sys.path.append(str(Path.cwd().parent))
from constants import PATH_TO_FINAL_OUTPUT
import statsmodels.api as sm


In [2]:
df_source = pd.read_csv(PATH_TO_FINAL_OUTPUT)
df_speaker_points = df_source.copy()


In [3]:
df_speaker_points['debate_date'] = pd.to_datetime(df_speaker_points['debate_date'])
df_speaker_points['speaker_first_debate_date'] = pd.to_datetime(df_speaker_points['speaker_first_debate_date'])

motion_balance = df_speaker_points[df_speaker_points['side'] == 'aff'].groupby(['debate_id', 'motion'])['ballots_gained'].mean().reset_index()
motion_balance = motion_balance.groupby('motion')['ballots_gained'].mean().reset_index()
motion_balance.columns = ['motion', 'motion_balance']

df_speaker_points = df_speaker_points.merge(motion_balance, on='motion', how='left')

df_speaker_points['years_since_first_debate'] = df_speaker_points['debate_date'].dt.year - df_speaker_points['speaker_first_debate_date'].dt.year

df_speaker_points['tournament_round'] = df_speaker_points.groupby('tournament_id')['debate_date'].rank(method='dense').astype(int)

df_speaker_points['is_aff'] = (df_speaker_points['side'] == 'aff').astype(int)
df_speaker_points['motion_balance_x_aff'] = df_speaker_points['motion_balance'] * df_speaker_points['is_aff']

In [4]:
def get_teammate_avg(group):
    teammate_avgs = []
    for idx in group.index:
        other_scores = group.loc[group.index != idx, 'speaker_points']
        teammate_avgs.append(other_scores.mean())
    return pd.Series(teammate_avgs, index=group.index)

df_speaker_points['avg_teammate_score'] = df_speaker_points.groupby(['debate_id', 'side'], group_keys=False).apply(get_teammate_avg)

In [5]:
df_reg = df_speaker_points.dropna(subset=['speaker_points', 'is_male', 'years_since_first_debate', 
                            'tournament_round', 'motion_balance_x_aff', 'motion_balance', 
                            'speaker_name', 'avg_teammate_score'])

X_pooled = df_reg[['is_male', 'years_since_first_debate', 'tournament_round', 
                    'motion_balance_x_aff', 'motion_balance', 'avg_teammate_score']].astype(float)
y_pooled = df_reg['speaker_points'].astype(float)

X_pooled = sm.add_constant(X_pooled)
model_pooled = sm.OLS(y_pooled, X_pooled).fit()

In [6]:
df_panel = df_reg.copy()
df_panel['speaker_id'] = pd.Categorical(df_panel['speaker_name']).codes
df_panel = df_panel.set_index(['speaker_id', 'debate_date'])

y_panel = df_panel['speaker_points']
# we have to drop years since first debate and is_male to use the fixed effects model
X_panel = df_panel[['tournament_round', 
                     'motion_balance_x_aff', 'motion_balance', 'avg_teammate_score']].astype(float)


model_fixed_effects = PanelOLS(y_panel, X_panel, entity_effects=True).fit()

In [7]:
print("="*80)
print("Pooled OLS Regression Results")
print("="*80)
print(model_pooled.summary())

print("\n" + "="*80)
print("Fixed Effects Panel Regression Results (Speaker Fixed Effects)")
print("="*80)
print(model_fixed_effects.summary)


print("\n" + "="*80)
print("Model Comparison")
print("="*80)
print(f"Pooled OLS R-squared: {model_pooled.rsquared:.4f}")
print(f"Fixed Effects R-squared: {model_fixed_effects.rsquared:.4f}")
print(f"Fixed Effects R-squared (within): {model_fixed_effects.rsquared_within:.4f}")

Pooled OLS Regression Results
                            OLS Regression Results                            
Dep. Variable:         speaker_points   R-squared:                       0.586
Model:                            OLS   Adj. R-squared:                  0.582
Method:                 Least Squares   F-statistic:                     155.8
Date:                Sun, 25 Jan 2026   Prob (F-statistic):          6.62e-123
Time:                        22:58:31   Log-Likelihood:                -1695.9
No. Observations:                 667   AIC:                             3406.
Df Residuals:                     660   BIC:                             3437.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

# Results


## Part 1: Individual Speaker Performance

### Pooled OLS Model (R² = 0.586)

**Significant Findings:**

- **Teammate Quality** (β = 0.746, p < 0.001): The strongest predictor. A 1-point increase in average teammate score predicts a 0.75-point increase in individual speaker points. This suggests strong within-team dynamics or clustering of talent.

- **Debater Experience** (β = 0.591, p < 0.001): Each additional year since first debate predicts a 0.59-point increase in speaker scores, indicating meaningful skill development over time.

**Non-Significant Findings:**

- **Gender** (β = -0.048, p = 0.842): No evidence of a gender performance gap
- **Tournament Round** (β = 0.009, p = 0.926): No warm-up effect detected
- **Motion Balance × Affirmative** (β = 0.020, p = 0.892): Topic balance does not favor either side
- **Motion Balance** (β = -0.358, p = 0.361): Overall motion balance shows no significant effect

### Fixed Effects Panel Model (Within R² = 0.394)

When controlling for individual fixed effects (time-invariant speaker characteristics), the model shows:

**Significant Findings:**

- **Teammate Quality** (β = 0.607, p < 0.001): Remains highly significant, though slightly smaller than in pooled OLS

**Non-Significant Findings:**

- **Tournament Round** (β = -0.036, p = 0.648): Still no warm-up effect
- **Motion Balance Effects**: Neither interaction term nor main effect shows significance

**Note**: Experience and gender effects cannot be estimated in the fixed effects model as they are absorbed by speaker-specific intercepts (time-invariant or slow-changing characteristics).

**Model Comparison:**

The F-test for poolability (F = 4.73, p < 0.001) strongly rejects the pooled model in favor of fixed effects, suggesting that individual speaker characteristics matter substantially. The high between-R² (0.849) indicates that speaker identity explains much of the variation in scores.

In [8]:
print("\n" + "="*80)
print("PART 2: Team-Level Analysis")
print("="*80)


PART 2: Team-Level Analysis


In [9]:
df_team = df_reg.copy()

df_team['prop_male'] = df_team.groupby(['debate_id', 'side'])['is_male'].transform('mean')

df_team['avg_team_experience'] = df_team.groupby(['debate_id', 'side'])['years_since_first_debate'].transform('mean')

team_debate = df_team.groupby(['debate_id', 'side']).agg({
    'ballots_gained': 'first',
    'prop_male': 'first',
    'avg_team_experience': 'first',
    'speaker_name': 'count'
}).reset_index()

team_debate.columns = ['debate_id', 'side', 'ballots_gained', 'prop_male', 'avg_team_experience', 'team_size']

team_stats = team_debate.groupby('side').agg({
    'ballots_gained': 'sum',
    'debate_id': 'count'
}).reset_index()

team_stats.columns = ['side', 'total_ballots', 'num_debates']
team_debate = team_debate.merge(team_stats, on='side', how='left')

team_debate['ballots_per_debate'] = team_debate['total_ballots'] / team_debate['num_debates'] / 3

team_debate_clean = team_debate.dropna(subset=['ballots_per_debate', 'prop_male', 'avg_team_experience'])


In [10]:
X_team = team_debate_clean[['prop_male', 'avg_team_experience']].astype(float)
y_team = team_debate_clean['ballots_per_debate'].astype(float)

X_team = sm.add_constant(X_team)
model_team = sm.OLS(y_team, X_team).fit()

print("\n" + "="*80)
print("Team-Level OLS Regression Results")
print("DV: Proportion of Debates Won Decisively (ballots_gained / num_debates / 3)")
print("="*80)
print(model_team.summary())


Team-Level OLS Regression Results
DV: Proportion of Debates Won Decisively (ballots_gained / num_debates / 3)
                            OLS Regression Results                            
Dep. Variable:     ballots_per_debate   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.003
Method:                 Least Squares   F-statistic:                    0.2245
Date:                Sun, 25 Jan 2026   Prob (F-statistic):              0.799
Time:                        22:58:31   Log-Likelihood:                 913.34
No. Observations:                 506   AIC:                            -1821.
Df Residuals:                     503   BIC:                            -1808.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
-----------


## Part 2: Team-Level Analysis

### Team Success Model (R² = 0.001)

We examine whether team composition predicts debate success, measured as the proportion of ballots won.

**Findings:**

- **Gender Composition** (β = -0.001, p = 0.816): No relationship between the proportion of male debaters and team success
- **Team Experience** (β = -0.001, p = 0.523): Average team experience does not predict winning

The extremely low R² (0.001) indicates that observable team characteristics explain virtually none of the variation in debate outcomes.


# Conclusions

**Key Takeaways:**

1. **Teammate quality dominates**: The strongest predictor of individual performance is having high-scoring teammates, explaining far more variance than individual characteristics

2. **Experience matters, but slowly**: Debaters improve over time (0.59 points per year), suggesting learning effects

3. **No gender gap**: We find no evidence of gender-based performance differences at either individual or team levels

4. **No strategic advantages**: Motion balance, side assignment, and tournament round show no systematic effects

5. **Team composition doesn't predict outcomes**: At the team level, neither gender mix nor experience explains success, suggesting that within-debate factors (argumentation quality, judge preferences) dominate

**Limitations:**

- Gender estimation from names may contain errors
- School effects are proxied imperfectly by teammate scores
- Unobserved factors (judge quality, room conditions, motion difficulty) may introduce noise
- The team-level analysis may be underpowered given the limited variation in outcomes

**Implications:**

The dominance of teammate effects suggests that debate success is highly contextual and relational rather than purely individual. The lack of systematic biases (gender, motion balance) suggests that the debate format provides relatively fair evaluation conditions.